# New Relic Service Map — Tests

Simple smoke/unit tests for `newrelic_service_map.py`.  
Cells 1–3 use `unittest.mock` so no real API key is needed.  
Cell 4 is an optional live smoke test — set `NR_API_KEY` in your environment to run it.

In [ ]:
# Cell 1 — Setup: add parent dir to path and import module
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from unittest.mock import patch, MagicMock
import newrelic_service_map as nrsm

FAKE_KEY = "FAKE_API_KEY_FOR_TESTS"
print("Import OK")

In [ ]:
# Cell 2 — Test search_entities_by_tag returns a list of entity dicts

MOCK_SEARCH_RESPONSE = {
    "data": {
        "actor": {
            "entitySearch": {
                "results": {
                    "entities": [
                        {
                            "name": "payments-service",
                            "guid": "GUID_1",
                            "entityType": "APM_APPLICATION_ENTITY",
                            "tags": [{"key": "environment", "values": ["production"]}],
                        },
                        {
                            "name": "orders-service",
                            "guid": "GUID_2",
                            "entityType": "APM_APPLICATION_ENTITY",
                            "tags": [{"key": "environment", "values": ["production"]}],
                        },
                    ]
                }
            }
        }
    }
}

mock_resp = MagicMock()
mock_resp.json.return_value = MOCK_SEARCH_RESPONSE
mock_resp.raise_for_status = MagicMock()

with patch("requests.post", return_value=mock_resp) as mock_post:
    entities = nrsm.search_entities_by_tag("environment", "production", FAKE_KEY)

assert isinstance(entities, list), "Expected a list"
assert len(entities) == 2, f"Expected 2 entities, got {len(entities)}"
assert entities[0]["name"] == "payments-service"
assert entities[0]["tags"] == {"environment": ["production"]}
assert mock_post.call_args[1]["headers"]["Api-Key"] == FAKE_KEY
print("PASS: search_entities_by_tag returned correct structure")

In [ ]:
# Cell 3 — Test extract_service_map returns expected graph shape

MOCK_REL_RESPONSE = {
    "data": {
        "actor": {
            "entity": {
                "name": "payments-service",
                "guid": "GUID_1",
                "entityType": "APM_APPLICATION_ENTITY",
                "tags": [],
                "relatedEntities": {
                    "results": [
                        {
                            "source": {"entity": {"name": "payments-service", "guid": "GUID_1", "entityType": "APM_APPLICATION_ENTITY"}},
                            "target": {"entity": {"name": "db-postgres", "guid": "GUID_DB", "entityType": "GENERIC_INFRASTRUCTURE_ENTITY"}},
                            "type": "CALLS",
                        }
                    ]
                },
            }
        }
    }
}

# Alternate mock for orders-service (no outbound calls)
MOCK_REL_RESPONSE_2 = {
    "data": {
        "actor": {
            "entity": {
                "name": "orders-service",
                "guid": "GUID_2",
                "entityType": "APM_APPLICATION_ENTITY",
                "tags": [],
                "relatedEntities": {"results": []},
            }
        }
    }
}

call_count = 0

def side_effect(*args, **kwargs):
    global call_count
    r = MagicMock()
    r.raise_for_status = MagicMock()
    # First call: entitySearch, next two: get_entity_relationships per entity
    if call_count == 0:
        r.json.return_value = MOCK_SEARCH_RESPONSE
    elif call_count == 1:
        r.json.return_value = MOCK_REL_RESPONSE
    else:
        r.json.return_value = MOCK_REL_RESPONSE_2
    call_count += 1
    return r

with patch("requests.post", side_effect=side_effect):
    graph = nrsm.extract_service_map("environment", "production", FAKE_KEY)

assert "entities" in graph, "graph must have 'entities' key"
assert "edges" in graph, "graph must have 'edges' key"
assert "GUID_1" in graph["entities"], "GUID_1 should be in entities"
assert "GUID_2" in graph["entities"], "GUID_2 should be in entities"
assert "GUID_DB" in graph["entities"], "Related entity GUID_DB should be added"
assert len(graph["edges"]) == 1, f"Expected 1 edge, got {len(graph['edges'])}"
assert graph["edges"][0]["source_guid"] == "GUID_1"
assert graph["edges"][0]["target_guid"] == "GUID_DB"
print("PASS: extract_service_map returns correct graph structure")
print(f"  Entities: {list(graph['entities'].keys())}")
print(f"  Edges: {graph['edges']}")

In [ ]:
# Cell 4 — Optional live smoke test (requires NR_API_KEY env var)

import os

# Optionally load from a .env file in the project root
try:
    from dotenv import load_dotenv
    load_dotenv(os.path.join("..", ".env"))
except ImportError:
    pass

api_key = os.environ.get("NR_API_KEY")

if not api_key:
    print("SKIP: NR_API_KEY not set — skipping live smoke test")
else:
    tag_key = os.environ.get("NR_TAG_KEY", "environment")
    tag_value = os.environ.get("NR_TAG_VALUE", "production")
    print(f"Running live search for tags.{tag_key} = '{tag_value}' ...")
    entities = nrsm.search_entities_by_tag(tag_key, tag_value, api_key)
    print(f"Found {len(entities)} entities")
    for e in entities[:5]:
        print(f"  - {e['name']} ({e['entityType']}) [{e['guid']}]")
    if entities:
        print("\nFetching relationships for first entity...")
        detail = nrsm.get_entity_relationships(entities[0]["guid"], api_key)
        print(f"  Relationships: {len(detail.get('relationships', []))}")
    print("PASS: live smoke test completed")

In [ ]:
# Cell 5 — Test search_entities_by_name

mock_resp = MagicMock()
mock_resp.json.return_value = MOCK_SEARCH_RESPONSE
mock_resp.raise_for_status = MagicMock()

with patch("requests.post", return_value=mock_resp) as mock_post:
    entities = nrsm.search_entities_by_name(["payments-service", "orders-service"], FAKE_KEY)

assert isinstance(entities, list), "Expected a list"
assert len(entities) == 2, f"Expected 2 entities, got {len(entities)}"
assert entities[0]["name"] == "payments-service"
assert entities[1]["name"] == "orders-service"

# Verify the query sent to NerdGraph contains name IN (
query_sent = mock_post.call_args[1]["json"]["query"]
assert "name IN (" in query_sent, f"Query should use name IN (...), got: {query_sent}"
assert "'payments-service'" in query_sent
assert "'orders-service'" in query_sent

# Empty list short-circuit — must not make any HTTP call
with patch("requests.post", side_effect=AssertionError("should not call API")) as mock_no_call:
    result = nrsm.search_entities_by_name([], FAKE_KEY)
assert result == [], f"Expected [] for empty names, got {result}"

print("PASS: search_entities_by_name returned correct structure and respects empty-list short-circuit")

In [ ]:
# Cell 6 — Test get_dependencies_for_services returns expected graph shape

dep_call_count = 0

def dep_side_effect(*args, **kwargs):
    global dep_call_count
    r = MagicMock()
    r.raise_for_status = MagicMock()
    # Call 0: entitySearch by name, calls 1-2: get_entity_relationships per entity
    if dep_call_count == 0:
        r.json.return_value = MOCK_SEARCH_RESPONSE
    elif dep_call_count == 1:
        r.json.return_value = MOCK_REL_RESPONSE
    else:
        r.json.return_value = MOCK_REL_RESPONSE_2
    dep_call_count += 1
    return r

with patch("requests.post", side_effect=dep_side_effect):
    dep_graph = nrsm.get_dependencies_for_services(["payments-service", "orders-service"], FAKE_KEY)

assert "entities" in dep_graph, "graph must have 'entities' key"
assert "edges" in dep_graph, "graph must have 'edges' key"
assert "GUID_1" in dep_graph["entities"], "GUID_1 should be in entities"
assert "GUID_2" in dep_graph["entities"], "GUID_2 should be in entities"
assert "GUID_DB" in dep_graph["entities"], "Related entity GUID_DB should be added"
assert len(dep_graph["edges"]) == 1, f"Expected 1 edge, got {len(dep_graph['edges'])}"
assert dep_graph["edges"][0]["source_guid"] == "GUID_1"
assert dep_graph["edges"][0]["target_guid"] == "GUID_DB"
print("PASS: get_dependencies_for_services returns correct graph structure")
print(f"  Entities: {list(dep_graph['entities'].keys())}")
print(f"  Edges: {dep_graph['edges']}")

In [ ]:
# Cell 7 — Test analyze_impact (pure in-memory, no API calls)
#
# Graph topology:
#   D → A → B   (D calls A, A calls B)
#   C → B       (C calls B)
#
# If B goes down:
#   direct     = {A, C}  (both directly call B)
#   transitive = {D}     (calls A, which calls B)
#   all_impacted = {A, C, D}

IMPACT_GRAPH = {
    "entities": {
        "A": {"name": "svc-a", "guid": "A", "entityType": "APM_APPLICATION_ENTITY", "tags": {}},
        "B": {"name": "svc-b", "guid": "B", "entityType": "APM_APPLICATION_ENTITY", "tags": {}},
        "C": {"name": "svc-c", "guid": "C", "entityType": "APM_APPLICATION_ENTITY", "tags": {}},
        "D": {"name": "svc-d", "guid": "D", "entityType": "APM_APPLICATION_ENTITY", "tags": {}},
    },
    "edges": [
        {"source_guid": "A", "target_guid": "B", "type": "CALLS"},
        {"source_guid": "C", "target_guid": "B", "type": "CALLS"},
        {"source_guid": "D", "target_guid": "A", "type": "CALLS"},
    ],
}

result = nrsm.analyze_impact(IMPACT_GRAPH, "B")

assert result["target"]["guid"] == "B", "target should be B"
direct_guids = {e["guid"] for e in result["direct"]}
transitive_guids = {e["guid"] for e in result["transitive"]}
all_guids = {e["guid"] for e in result["all_impacted"]}

assert direct_guids == {"A", "C"}, f"Expected direct={{A,C}}, got {direct_guids}"
assert transitive_guids == {"D"}, f"Expected transitive={{D}}, got {transitive_guids}"
assert all_guids == {"A", "C", "D"}, f"Expected all_impacted={{A,C,D}}, got {all_guids}"

# No upstream callers — D has nothing calling it
result_d = nrsm.analyze_impact(IMPACT_GRAPH, "D")
assert result_d["direct"] == [], f"Expected no direct callers for D, got {result_d['direct']}"
assert result_d["transitive"] == [], "Expected no transitive callers for D"

# Cycle safety: A → B → A cycle should not loop forever
CYCLE_GRAPH = {
    "entities": {
        "A": {"name": "svc-a", "guid": "A", "entityType": "APM_APPLICATION_ENTITY", "tags": {}},
        "B": {"name": "svc-b", "guid": "B", "entityType": "APM_APPLICATION_ENTITY", "tags": {}},
    },
    "edges": [
        {"source_guid": "A", "target_guid": "B", "type": "CALLS"},
        {"source_guid": "B", "target_guid": "A", "type": "CALLS"},
    ],
}
cycle_result = nrsm.analyze_impact(CYCLE_GRAPH, "B")
assert {e["guid"] for e in cycle_result["all_impacted"]} == {"A"}, "Cycle graph: only A impacts B"

# Unknown guid raises KeyError
try:
    nrsm.analyze_impact(IMPACT_GRAPH, "NONEXISTENT")
    assert False, "Should have raised KeyError"
except KeyError:
    pass

print("PASS: analyze_impact correctly identifies direct, transitive, and all impacted services")
print(f"  If svc-b fails → direct: {[e['name'] for e in result['direct']]}, "
      f"transitive: {[e['name'] for e in result['transitive']]}")